# 04 (Colab) · Multi-view reconstruction of your own photos + Blender export
Same as `04_demo_export`, for **Google Colab** (no Kaggle queue). Original DA3, released weights,
feed-forward, no training. See docs/KAGGLE_EXAMPLES.md §8.

**Before running**
1. **Runtime → Change runtime type → T4 GPU** (this notebook asks for it; check it is selected).
2. Private repo only: left bar → **key icon (Secrets)** → add `GITHUB_TOKEN` (the same token as on
   Kaggle) and switch on **Notebook access**.
3. Your photos, one of the two:
   * **Google Drive** (recommended): put the folder (e.g. `building/` with the JPGs) in your Drive,
     e.g. `MyDrive/DA3/building`, and set `MY_MULTIVIEW` below;
   * **upload a .zip** of the folder from your computer (`PHOTOS_FROM = "upload"`).
4. **Runtime → Run all**.

Results: `/content/demo/<name>/` and `<name>.zip`. Colab deletes `/content` when the session ends,
so the last cell copies the zips to Drive (`SAVE_TO_DRIVE`) or downloads them.

In [ ]:
import torch
!nvidia-smi -L
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then Runtime > Run all again"

In [ ]:
# --- repo setup (Google Colab) -----------------------------------------------------------------
REPO_URL = "https://github.com/davideCabitz/DA3_LAB_Advanced_CV.git"   # push your branch first
BRANCH = "main"
import os, subprocess
os.environ["GIT_TERMINAL_PROMPT"] = "0"   # fail fast instead of waiting for a password
os.chdir("/content")
url = REPO_URL
try:  # private repo: left bar > key icon (Secrets) > add GITHUB_TOKEN, switch on "Notebook access"
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    print("no GITHUB_TOKEN secret: cloning without a token (works only for a public repo)")
if not os.path.exists("DA3_LAB_Advanced_CV"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "DA3_LAB_Advanced_CV"], check=True)
else:
    subprocess.run(["git", "-C", "DA3_LAB_Advanced_CV", "pull", "--ff-only"], check=True)
os.chdir("/content/DA3_LAB_Advanced_CV")
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
!git log -1 --format="commit %H  %cd"

In [ ]:
# Open3D (TSDF meshing) + self-test: if TSDF fusion is broken here, infer.py uses a simpler mesher.
# 0.19.0 is the version whose legacy TSDF works (0.20.0 returns empty meshes); it exists up to Python 3.12.
import sys
if sys.version_info[:2] <= (3, 12):
    !pip install -q "open3d==0.19.0"
else:
    !pip install -q "open3d>=0.19.0"
!python scripts/check_open3d.py --out /content/open3d_check.json

### Your inputs

In [ ]:
PHOTOS_FROM = "drive"   # "drive": a folder in your Google Drive | "upload": a .zip from your computer
MY_MULTIVIEW = "/content/drive/MyDrive/DA3/building"   # ONE scene (photos directly inside) or a folder of scene sub-folders
MY_SINGLE = ""          # optional: folder of independent photos -> depth + ray map per photo
WEIGHTS = "official:da3-base"            # released weights (Apache-2.0); "official:da3-small" is faster
EXTRA_ARGS = "--conf-percentile 60"      # buildings: fewer sky / low-confidence points; add "--max-views 60" if CUDA out of memory
SAVE_TO_DRIVE = "/content/drive/MyDrive/DA3/results"   # "" = download the zips instead
RUN_EXAMPLES = False    # True: also reconstruct the two built-in examples (SOH, robot_unitree)
OUT = "/content/demo"

if PHOTOS_FROM == "drive" or SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if PHOTOS_FROM == "upload":
    import io, zipfile
    from google.colab import files
    uploaded = files.upload()   # choose the .zip of your photo folder
    for fname, data in uploaded.items():
        zipfile.ZipFile(io.BytesIO(data)).extractall("/content/my_photos")
    MY_MULTIVIEW = "/content/my_photos"

from pathlib import Path
IMG = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
def has_images(d):
    return any(p.suffix.lower() in IMG and not p.name.startswith(".") for p in Path(d).iterdir())
def scenes_in(folder):
    f = Path(folder)
    assert f.exists(), f"not found: {folder} (check the path in the Files panel on the left)"
    if has_images(f):
        return {f.name: str(f)}
    subs = [d for d in sorted(f.iterdir()) if d.is_dir() and not d.name.startswith(("__MACOSX", "."))]
    found = {d.name: str(d) for d in subs if has_images(d)}
    if not found:  # e.g. a zip containing one folder that itself contains the scene folders
        for d in subs:
            found.update({e.name: str(e) for e in sorted(d.iterdir()) if e.is_dir() and has_images(e)})
    return found

sets = {}
if RUN_EXAMPLES:
    if not os.path.exists("/content/Depth-Anything-3"):
        subprocess.run(["git", "clone", "https://github.com/ByteDance-Seed/Depth-Anything-3.git", "/content/Depth-Anything-3"], check=True)
    !python scripts/extract_frames.py /content/Depth-Anything-3/assets/examples/robot_unitree.mp4 \
        /content/samples/robot_unitree --frames 12 --start 0.0 --end 0.6
    sets.update({"SOH": "/content/Depth-Anything-3/assets/examples/SOH", "robot_unitree": "/content/samples/robot_unitree"})
if MY_MULTIVIEW:
    sets.update({f"my_{k}": v for k, v in scenes_in(MY_MULTIVIEW).items()})
for k, v in sets.items():
    print(f"{k}: {sum(1 for p in Path(v).iterdir() if p.suffix.lower() in IMG)} photos in {v}")
assert sets or MY_SINGLE, "no photos found: check MY_MULTIVIEW"

### Multi-view reconstructions

In [ ]:
for name, folder in sets.items():
    !python scripts/infer.py --images "{folder}" --out "{OUT}/{name}" --weights {WEIGHTS} {EXTRA_ARGS}
    !python scripts/render_preview.py "{OUT}/{name}"
!ls -la {OUT}

In [ ]:
from IPython.display import Image, display
for name in sets:
    print(name, "| columns: RGB | depth (red = near, blue = far) | ray map (direction as RGB) | depth confidence (red = high, blue = low)")
    for f in ("preview.png", "preview_3d.png"):
        p = f"{OUT}/{name}/{f}"
        if os.path.exists(p):
            display(Image(p, width=1000))
    s = f"{OUT}/{name}/summary.json"
    if os.path.exists(s):
        print(open(s).read()[:1500])

### Optional: single images (depth map + ray map per photo, N_v = 1)

In [ ]:
if MY_SINGLE:
    !python scripts/infer.py --images "{MY_SINGLE}" --out {OUT}/single --weights {WEIGHTS} --per-image --mesh none
    for d in sorted(Path(f"{OUT}/single").iterdir()):
        if d.is_dir():
            print(d.name); display(Image(str(d / "preview.png"), width=1000))
else:
    print("set MY_SINGLE to run this part")

### Save the results (Colab deletes /content at the end of the session)

In [ ]:
import glob, shutil
zips = sorted(glob.glob(f"{OUT}/*.zip"))
print(zips)
if SAVE_TO_DRIVE:
    os.makedirs(SAVE_TO_DRIVE, exist_ok=True)
    for z in zips:
        shutil.copy(z, SAVE_TO_DRIVE)
        print("copied to Drive:", os.path.join(SAVE_TO_DRIVE, os.path.basename(z)))
else:
    from google.colab import files
    for z in zips:
        files.download(z)